In [1]:
# ── Cell 1: mount drive & confirm GPU ──
from google.colab import drive
drive.mount('/content/drive')

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

Mounted at /content/drive
CUDA available: True
Device name: NVIDIA L4


In [2]:
!pip install -q --upgrade transformers accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 157.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.5 MB/s eta 0:00:00


In [2]:
import copy
import json

import numpy as np
import torch
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import f1_score

from tqdm.auto import tqdm

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=3
).to(device)

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
LABEL2ID = {"e": 0, "n": 1, "c": 2}
BASE     = "drive/MyDrive/anli/"

def load_jsonl(filepath):
    with open(filepath) as f:
        return [json.loads(l) for l in open(filepath) if l.strip()]

class ANLIDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        row = self.data[idx]
        enc = tokenizer(row["context"], row["hypothesis"],
                        max_length=128, truncation=True,
                        padding="max_length", return_tensors="pt")
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(LABEL2ID[row["label"]], dtype=torch.long)
        }

r1_data = load_jsonl(BASE + "R1/" + "train.jsonl")
r2_data = load_jsonl(BASE + "R2/" + "train.jsonl")
r3_data = load_jsonl(BASE + "R3/" + "train.jsonl")
r1_dev  = load_jsonl(BASE + "R1/" + "dev.jsonl")
r2_dev  = load_jsonl(BASE + "R2/" + "dev.jsonl")
r3_dev  = load_jsonl(BASE + "R3/" + "dev.jsonl")

print("R1:", len(r1_data), "| R2:", len(r2_data), "| R3:", len(r3_data))

R1: 16946 | R2: 45460 | R3: 100459


In [4]:
def evaluate(model, data, device):
    model.eval()

    loader = DataLoader(
        ANLIDataset(data),
        batch_size=32,
        shuffle=False
    )

    all_preds = []
    all_labels = []
    total_loss = 0

    with torch.no_grad():
        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            preds = outputs.logits.argmax(dim=-1)

            total_loss += outputs.loss.item()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = np.mean(np.array(all_preds) == np.array(all_labels))

    f1 = f1_score(
        all_labels,
        all_preds,
        average=None,
        labels=[0, 1, 2]
    )

    avg_loss = total_loss / len(loader)

    return avg_loss, acc, f1

In [7]:
# -------------------------
# Training Parameters
# -------------------------
MAX_EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# -------------------------
# Store results for graphs
# -------------------------
history = {
    "R1": [],
    "R2": [],
    "R3": []
}

# -------------------------
# Curriculum Learning
# -------------------------
for round_name, train_data, val_data in [

    ("R1", r1_data, r1_dev),
    ("R2", r2_data, r2_dev),
    ("R3", r3_data, r3_dev),

]:

    print(f"\n{'='*12} {round_name} {'='*12}")

    train_loader = DataLoader(
        ANLIDataset(train_data),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    total_steps = len(train_loader) * MAX_EPOCHS
    warmup_steps = int(0.1 * total_steps)

    optimizer = AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    best_val_loss = float("inf")
    best_model_state = None

    checkpoint_path = f"/content/best_{round_name}"

    print(
        f"{'Epoch':<8}"
        f"{'Train Loss':<15}"
        f"{'Val Loss':<15}"
        f"{'Accuracy':<12}"
        f"{'F1 Entail':<15}"
        f"{'F1 Neutral':<15}"
        f"{'F1 Contra'}"
    )

    # ==========================
    # Epoch Loop
    # ==========================
    for epoch in range(MAX_EPOCHS):

        model.train()

        running_loss = 0

        progress_bar = tqdm(
            train_loader,
            desc=f"{round_name} Epoch {epoch+1}/{MAX_EPOCHS}",
            dynamic_ncols=True
        )

        for batch in progress_bar:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            running_loss += loss.item()

            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}",
                lr=f"{scheduler.get_last_lr()[0]:.2e}"
            )

        # ==========================
        # End of Epoch Evaluation
        # ==========================

        train_loss = running_loss / len(train_loader)

        val_loss, accuracy, f1 = evaluate(
            model,
            val_data,
            device
        )

        # Store metrics for graphs
        history[round_name].append({

            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "accuracy": accuracy,
            "f1_entailment": f1[0],
            "f1_neutral": f1[1],
            "f1_contradiction": f1[2]

        })

        print(
            f"{epoch+1:<8}"
            f"{train_loss:<15.6f}"
            f"{val_loss:<15.6f}"
            f"{accuracy:<12.6f}"
            f"{f1[0]:<15.6f}"
            f"{f1[1]:<15.6f}"
            f"{f1[2]:.6f}"
        )

        # ==========================
        # Save Best Model
        # ==========================

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_model_state = copy.deepcopy(model.state_dict())

            model.save_pretrained(checkpoint_path)
            tokenizer.save_pretrained(checkpoint_path)

            print("✓ Best model updated.")

    # ==========================
    # Restore Best Model
    # ==========================

    print(f"\nLoading best checkpoint from {round_name}")

    model.load_state_dict(best_model_state)

# ==========================
# Save Final Model
# ==========================

model.save_pretrained("/content/drive/MyDrive/distilbert_curriculum")
tokenizer.save_pretrained("/content/drive/MyDrive/distilbert_curriculum")

print("\nTraining complete!")
print("Final model saved successfully.")


============ R1 ============
Epoch   Train Loss     Val Loss       Accuracy    F1 Entail      F1 Neutral     F1 Contra


R1 Epoch 1/3:   0%|          | 0/530 [00:00<?, ?it/s]

1       0.910308       1.216294       0.406000    0.428571       0.512301       0.221328


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Best model updated.


...

2       0.498726       1.488068       0.430833    0.415423       0.467120       0.403361


R3 Epoch 3/3:   0%|          | 0/3140 [00:00<?, ?it/s]

3       0.353829       1.732044       0.432500    0.427896       0.452071       0.414669

Loading best checkpoint from R3


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete!
Final model saved successfully.


In [8]:
import os
import shutil

# Destination in Google Drive
DRIVE_DIR = "/content/drive/MyDrive/distilbert_curriculum"
os.makedirs(DRIVE_DIR, exist_ok=True)

folders_to_copy = [
    "best_R1",
    "best_R2",
    "best_R3",
    "curriculum_model"
]

for folder in folders_to_copy:
    src = f"/content/{folder}"
    dst = f"{DRIVE_DIR}/{folder}"

    if os.path.exists(src):
        # Remove old copy if it exists
        if os.path.exists(dst):
            shutil.rmtree(dst)

        shutil.copytree(src, dst)
        print(f"✓ Copied {folder} to Google Drive")
    else:
        print(f"✗ {folder} not found")

print("\nAll available models have been copied successfully!")

✓ Copied best_R1 to Google Drive
✓ Copied best_R2 to Google Drive
✓ Copied best_R3 to Google Drive
✗ curriculum_model not found

All available models have been copied successfully!


In [9]:
r1_test = load_jsonl(BASE + "R1/" + "test.jsonl")
r2_test = load_jsonl(BASE + "R2/" + "test.jsonl")
r3_test = load_jsonl(BASE + "R3/" + "test.jsonl")

print(f"\n{'Split':<10}{'Test Loss':<14}{'Accuracy':<12}{'F1 Entail':<14}{'F1 Neutral':<14}{'F1 Contra'}")
print("-" * 72)

for split_name, test_data in [
    ("R1 test", r1_test),
    ("R2 test", r2_test),
    ("R3 test", r3_test),
]:
    loss, acc, f1 = evaluate(model, test_data, device)
    print(f"{split_name:<10}{loss:<14.4f}{acc:<12.4f}{f1[0]:<14.4f}{f1[1]:<14.4f}{f1[2]:.4f}")


Split     Test Loss     Accuracy    F1 Entail     F1 Neutral    F1 Contra
------------------------------------------------------------------------
R1 test   1.1129        0.5160      0.5463        0.5609        0.4157
R2 test   1.3715        0.4110      0.4714        0.4274        0.2907
R3 test   1.2678        0.4483      0.4968        0.4476        0.3844


In [10]:
ID2LABEL = {0: "entailment", 1: "neutral", 2: "contradiction"}

def high_confidence_wrong(model, data, device, threshold=0.90):
    model.eval()
    loader = DataLoader(ANLIDataset(data), batch_size=32, shuffle=False)
    cases  = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"]
            outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
            probs          = F.softmax(outputs.logits, dim=-1).cpu()
            confs, preds   = probs.max(dim=-1)
            for i in range(len(labels)):
                if preds[i] != labels[i] and confs[i] >= threshold:
                    cases.append({
                        "predicted":  ID2LABEL[preds[i].item()],
                        "true_label": ID2LABEL[labels[i].item()],
                        "confidence": round(confs[i].item(), 4),
                    })
    return cases

for split_name, test_data in [
    ("R1", r1_test),
    ("R2", r2_test),
    ("R3", r3_test),
]:
    cases = high_confidence_wrong(model, test_data, device)
    print(f"{split_name}: {len(cases)} high-confidence mistakes out of {len(test_data)} examples")

R1: 30 high-confidence mistakes out of 1000 examples
R2: 76 high-confidence mistakes out of 1000 examples
R3: 62 high-confidence mistakes out of 1200 examples


In [11]:
history

{'R1': [{'epoch': 1,
   'train_loss': 0.9103082779443489,
   'val_loss': 1.2162944003939629,
   'accuracy': np.float64(0.406),
   'f1_entailment': np.float64(0.42857142857142855),
   'f1_neutral': np.float64(0.5123010130246021),
   'f1_contradiction': np.float64(0.22132796780684105)},
  {'epoch': 2,
   'train_loss': 0.6612348249498403,
   'val_loss': 1.2587324492633343,
   'accuracy': np.float64(0.416),
   'f1_entailment': np.float64(0.3872832369942196),
   'f1_neutral': np.float64(0.5080291970802919),
   'f1_contradiction': np.float64(0.3467094703049759)},
  {'epoch': 3,
   'train_loss': 0.5228945824897514,
   'val_loss': 1.3637669794261456,
   'accuracy': np.float64(0.418),
   'f1_entailment': np.float64(0.4117647058823529),
   'f1_neutral': np.float64(0.49850746268656715),
   'f1_contradiction': np.float64(0.33766233766233766)}],
 'R2': [{'epoch': 1,
   'train_loss': 0.6610956136019275,
   'val_loss': 1.3103897143155336,
   'accuracy': np.float64(0.392),
   'f1_entailment': np.float

In [13]:
# -------------------------
# Training Parameters
# -------------------------
MAX_EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# -------------------------
# Fresh model for baseline
# -------------------------
baseline_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=3
).to(device)

# -------------------------
# Mixed Dataset
# -------------------------
all_train = r1_data + r2_data + r3_data
all_dev   = r1_dev + r2_dev + r3_dev

print(f"Baseline training on {len(all_train)} mixed examples")

train_loader = DataLoader(
    ANLIDataset(all_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)

total_steps = len(train_loader) * MAX_EPOCHS
warmup_steps = int(0.1 * total_steps)

optimizer = AdamW(
    baseline_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

best_val_loss = float("inf")
best_model_state = None

checkpoint_path = "/content/best_baseline"

print("\n========== Baseline (Mixed Training) ==========\n")

print(
    f"{'Epoch':<8}"
    f"{'Train Loss':<15}"
    f"{'Val Loss':<15}"
    f"{'Accuracy':<12}"
    f"{'F1 Entail':<15}"
    f"{'F1 Neutral':<15}"
    f"{'F1 Contra'}"
)

# ==========================
# Epoch Loop
# ==========================

for epoch in range(MAX_EPOCHS):

    baseline_model.train()

    running_loss = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Baseline Epoch {epoch+1}/{MAX_EPOCHS}",
        dynamic_ncols=True
    )

    for batch in progress_bar:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = baseline_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            baseline_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        running_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}"
        )

    # ==========================
    # End of Epoch Evaluation
    # ==========================

    train_loss = running_loss / len(train_loader)

    val_loss, accuracy, f1 = evaluate(
        baseline_model,
        all_dev,
        device
    )

    print(
        f"{epoch+1:<8}"
        f"{train_loss:<15.6f}"
        f"{val_loss:<15.6f}"
        f"{accuracy:<12.6f}"
        f"{f1[0]:<15.6f}"
        f"{f1[1]:<15.6f}"
        f"{f1[2]:.6f}"
    )

    # ==========================
    # Save Best Model
    # ==========================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_model_state = copy.deepcopy(
            baseline_model.state_dict()
        )

        baseline_model.save_pretrained(checkpoint_path)
        tokenizer.save_pretrained(checkpoint_path)

        print("✓ Best model updated.")

# ==========================
# Restore Best Model
# ==========================

print("\nLoading best baseline checkpoint")

baseline_model.load_state_dict(best_model_state)

# ==========================
# Save Final Model
# ==========================

baseline_model.save_pretrained("/content/drive/MyDrive/baseline_model")
tokenizer.save_pretrained("/content/drive/MyDrive/baseline_model")

print("\nBaseline training complete!")
print("Final model saved successfully.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Baseline training on 162865 mixed examples

========== Baseline (Mixed Training) ==========

Epoch   Train Loss     Val Loss       Accuracy    F1 Entail      F1 Neutral     F1 Contra


Baseline Epoch 1/3:   0%|          | 0/5090 [00:00<?, ?it/s]

1       0.740561       1.274679       0.435312    0.436031       0.482224       0.380503


...

2       0.496276       1.383882       0.442188    0.476852       0.503952       0.307785


Baseline Epoch 3/3:   0%|          | 0/5090 [00:00<?, ?it/s]

3       0.361734       1.561218       0.456875    0.458700       0.511861       0.388371

Loading best baseline checkpoint


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Baseline training complete!
Final model saved successfully.


In [14]:
r1_test = load_jsonl(BASE + "R1/" + "test.jsonl")
r2_test = load_jsonl(BASE + "R2/" + "test.jsonl")
r3_test = load_jsonl(BASE + "R3/" + "test.jsonl")

print(f"\n{'Split':<10}{'Test Loss':<14}{'Accuracy':<12}{'F1 Entail':<14}{'F1 Neutral':<14}{'F1 Contra'}")
print("-" * 72)

for split_name, test_data in [
    ("R1 test", r1_test),
    ("R2 test", r2_test),
    ("R3 test", r3_test),
]:
    loss, acc, f1 = evaluate(baseline_model, test_data, device)
    print(f"{split_name:<10}{loss:<14.4f}{acc:<12.4f}{f1[0]:<14.4f}{f1[1]:<14.4f}{f1[2]:.4f}")


Split     Test Loss     Accuracy    F1 Entail     F1 Neutral    F1 Contra
------------------------------------------------------------------------
R1 test   1.1004        0.5220      0.5222        0.5728        0.4660
R2 test   1.3606        0.4120      0.4673        0.4219        0.3238
R3 test   1.3445        0.4200      0.4197        0.4560        0.3821


In [15]:
ID2LABEL = {0: "entailment", 1: "neutral", 2: "contradiction"}

def high_confidence_wrong(model, data, device, threshold=0.90):
    model.eval()
    loader = DataLoader(ANLIDataset(data), batch_size=32, shuffle=False)
    cases  = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"]
            outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
            probs          = F.softmax(outputs.logits, dim=-1).cpu()
            confs, preds   = probs.max(dim=-1)
            for i in range(len(labels)):
                if preds[i] != labels[i] and confs[i] >= threshold:
                    cases.append({
                        "predicted":  ID2LABEL[preds[i].item()],
                        "true_label": ID2LABEL[labels[i].item()],
                        "confidence": round(confs[i].item(), 4),
                    })
    return cases

for split_name, test_data in [
    ("R1", r1_test),
    ("R2", r2_test),
    ("R3", r3_test),
]:
    cases = high_confidence_wrong(baseline_model, test_data, device)
    print(f"{split_name}: {len(cases)} high-confidence mistakes out of {len(test_data)} examples")

R1: 39 high-confidence mistakes out of 1000 examples
R2: 91 high-confidence mistakes out of 1000 examples
R3: 89 high-confidence mistakes out of 1200 examples


In [5]:
# -------------------------
# Training Parameters
# -------------------------
MAX_EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# -------------------------
# Fresh model for baseline
# -------------------------
baseline_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=3
).to(device)

# -------------------------
# Mixed Dataset
# -------------------------
all_train = r2_data
all_dev   = r2_dev

print(f"Baseline training on {len(all_train)} mixed examples")

train_loader = DataLoader(
    ANLIDataset(all_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)

total_steps = len(train_loader) * MAX_EPOCHS
warmup_steps = int(0.1 * total_steps)

optimizer = AdamW(
    baseline_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

best_val_loss = float("inf")
best_model_state = None

checkpoint_path = "/content/best_baseline"

print("\n========== Baseline (Mixed Training) ==========\n")

print(
    f"{'Epoch':<8}"
    f"{'Train Loss':<15}"
    f"{'Val Loss':<15}"
    f"{'Accuracy':<12}"
    f"{'F1 Entail':<15}"
    f"{'F1 Neutral':<15}"
    f"{'F1 Contra'}"
)

# ==========================
# Epoch Loop
# ==========================

for epoch in range(MAX_EPOCHS):

    baseline_model.train()

    running_loss = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Baseline Epoch {epoch+1}/{MAX_EPOCHS}",
        dynamic_ncols=True
    )

    for batch in progress_bar:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = baseline_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            baseline_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        running_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}"
        )

    # ==========================
    # End of Epoch Evaluation
    # ==========================

    train_loss = running_loss / len(train_loader)

    val_loss, accuracy, f1 = evaluate(
        baseline_model,
        all_dev,
        device
    )

    print(
        f"{epoch+1:<8}"
        f"{train_loss:<15.6f}"
        f"{val_loss:<15.6f}"
        f"{accuracy:<12.6f}"
        f"{f1[0]:<15.6f}"
        f"{f1[1]:<15.6f}"
        f"{f1[2]:.6f}"
    )

    # ==========================
    # Save Best Model
    # ==========================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_model_state = copy.deepcopy(
            baseline_model.state_dict()
        )

        baseline_model.save_pretrained(checkpoint_path)
        tokenizer.save_pretrained(checkpoint_path)

        print("✓ Best model updated.")

# ==========================
# Restore Best Model
# ==========================

print("\nLoading best baseline checkpoint")

baseline_model.load_state_dict(best_model_state)

# ==========================
# Save Final Model
# ==========================

baseline_model.save_pretrained("/content/drive/MyDrive/distilbert_r2_model")
tokenizer.save_pretrained("/content/drive/MyDrive/distilbert_r2_model")

print("\nBaseline training complete!")
print("Final model saved successfully.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Baseline training on 45460 mixed examples

========== Baseline (Mixed Training) ==========

Epoch   Train Loss     Val Loss       Accuracy    F1 Entail      F1 Neutral     F1 Contra


Baseline Epoch 1/3:   0%|          | 0/1421 [00:00<?, ?it/s]

1       0.749969       1.344570       0.400000    0.365682       0.512690       0.271457


...

2       0.487808       1.429134       0.417000    0.419562       0.477143       0.332696


Baseline Epoch 3/3:   0%|          | 0/1421 [00:00<?, ?it/s]

3       0.356837       1.605219       0.437000    0.419486       0.504249       0.374775

Loading best baseline checkpoint


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Baseline training complete!
Final model saved successfully.


In [8]:
# -------------------------
# Training Parameters
# -------------------------
MAX_EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# -------------------------
# Fresh model for baseline
# -------------------------
baseline_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=3
).to(device)

# -------------------------
# Mixed Dataset
# -------------------------
all_train = r3_data
all_dev   = r3_dev

print(f"Baseline training on {len(all_train)} mixed examples")

train_loader = DataLoader(
    ANLIDataset(all_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)

total_steps = len(train_loader) * MAX_EPOCHS
warmup_steps = int(0.1 * total_steps)

optimizer = AdamW(
    baseline_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

best_val_loss = float("inf")
best_model_state = None

checkpoint_path = "/content/best_baseline"

print("\n========== Baseline (Mixed Training) ==========\n")

print(
    f"{'Epoch':<8}"
    f"{'Train Loss':<15}"
    f"{'Val Loss':<15}"
    f"{'Accuracy':<12}"
    f"{'F1 Entail':<15}"
    f"{'F1 Neutral':<15}"
    f"{'F1 Contra'}"
)

# ==========================
# Epoch Loop
# ==========================

for epoch in range(MAX_EPOCHS):

    baseline_model.train()

    running_loss = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Baseline Epoch {epoch+1}/{MAX_EPOCHS}",
        dynamic_ncols=True
    )

    for batch in progress_bar:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = baseline_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            baseline_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        running_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}"
        )

    # ==========================
    # End of Epoch Evaluation
    # ==========================

    train_loss = running_loss / len(train_loader)

    val_loss, accuracy, f1 = evaluate(
        baseline_model,
        all_dev,
        device
    )

    print(
        f"{epoch+1:<8}"
        f"{train_loss:<15.6f}"
        f"{val_loss:<15.6f}"
        f"{accuracy:<12.6f}"
        f"{f1[0]:<15.6f}"
        f"{f1[1]:<15.6f}"
        f"{f1[2]:.6f}"
    )

    # ==========================
    # Save Best Model
    # ==========================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_model_state = copy.deepcopy(
            baseline_model.state_dict()
        )

        baseline_model.save_pretrained(checkpoint_path)
        tokenizer.save_pretrained(checkpoint_path)

        print("✓ Best model updated.")

# ==========================
# Restore Best Model
# ==========================

print("\nLoading best baseline checkpoint")

baseline_model.load_state_dict(best_model_state)

# ==========================
# Save Final Model
# ==========================

baseline_model.save_pretrained("/content/drive/MyDrive/distilbert_r3_model")
tokenizer.save_pretrained("/content/drive/MyDrive/distilbert_r3_model")

print("\nBaseline training complete!")
print("Final model saved successfully.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Baseline training on 100459 mixed examples

========== Baseline (Mixed Training) ==========

Epoch   Train Loss     Val Loss       Accuracy    F1 Entail      F1 Neutral     F1 Contra


Baseline Epoch 1/3:   0%|          | 0/3140 [00:00<?, ?it/s]

1       0.786703       1.345618       0.410833    0.402036       0.470226       0.331250


...

2       0.526203       1.447240       0.435833    0.468508       0.476404       0.327273


Baseline Epoch 3/3:   0%|          | 0/3140 [00:00<?, ?it/s]

3       0.382726       1.633899       0.438333    0.457607       0.465817       0.378698

Loading best baseline checkpoint


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Baseline training complete!
Final model saved successfully.
